In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from sklearn import metrics

In [ ]:
# ============ CHANGE THESE ============
main_dir = '/home/user/Rotarod'
pickle_dir = main_dir + '/results/pickle/'
plots_dir  = main_dir + '/results/plots/'

frame_rate = 30
# ======================================

Path(plots_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
def data_for_clustering(df, exclude=('tailtip', 'nose')):
    """Stack all body part (x, y) positions into an array for silhouette scoring.

    Returns X (N x 2) and labels (N,) where each label is the body part index.
    Points with NaN in either coordinate are dropped per body part.
    Body parts listed in `exclude` are skipped.
    """
    bodyparts = [bp for bp in df.columns.get_level_values(0).unique() if bp not in exclude]

    X_list, label_list = [], []
    for i, bp in enumerate(bodyparts):
        sub = df[bp][['x', 'y']].dropna()
        if len(sub) == 0:
            continue
        X_list.append(sub.to_numpy())
        label_list.append(np.full(len(sub), i))

    if not X_list:
        return np.empty((0, 2)), np.array([])

    return np.vstack(X_list), np.concatenate(label_list)

In [ ]:
# Discover all individual pickle files (one per mouse/video)
# Skip the combined all_results.pickle
pickle_files = sorted(Path(pickle_dir).glob('*.pickle'))
pickle_files = [p for p in pickle_files if p.stem != 'all_results' and p.stem != 'ctr_h3_silh_results']

mouse_names = [p.stem for p in pickle_files]
print(f'Found {len(pickle_files)} mice:')
for name in mouse_names:
    print(f'  {name}')

In [ ]:
# Compute silhouette score for each mouse (one score per video/trial)
silhouette_score_dict = {}

for mouse_name, pkl_path in zip(mouse_names, pickle_files):
    print(f'Processing {mouse_name} ...')

    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    df = data['df']

    X, labels = data_for_clustering(df)

    n_labels = len(np.unique(labels))
    if X.shape[0] < 2 or n_labels < 2:
        print(f'  WARNING: not enough data to compute silhouette score')
        silhouette_score_dict[mouse_name] = np.nan
        continue

    s_score = metrics.silhouette_score(X, labels)
    silhouette_score_dict[mouse_name] = s_score
    print(f'  Silhouette score: {s_score:.4f}')

print('\nDone.')

In [ ]:
# Plot body part scatter for each mouse with silhouette score
for mouse_name, pkl_path in zip(mouse_names, pickle_files):
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    df = data['df']
    bodyparts = [bp for bp in df.columns.get_level_values(0).unique()
                 if bp not in ('tailtip', 'nose')]

    X, labels = data_for_clustering(df)
    if X.shape[0] == 0:
        continue

    fig, ax = plt.subplots(figsize=(5, 5))
    for i, bp in enumerate(bodyparts):
        mask = labels == i
        ax.scatter(X[mask, 0], X[mask, 1], label=bp, s=2, alpha=0.3)

    ax.invert_yaxis()
    s_score = silhouette_score_dict[mouse_name]
    ax.set_title(f'{mouse_name}  —  Silhouette: {s_score:.3f}')
    ax.set_xlabel('x (px)')
    ax.set_ylabel('y (px)')
    ax.legend(markerscale=5, fontsize=9)
    fig.tight_layout()

    fig_path = plots_dir + f'scatter_{mouse_name}.png'
    fig.savefig(fig_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'  Saved -> {fig_path}')

In [ ]:
# ============================================================
# Assign each mouse to a group: 'ctr' (control) or 'h3'
# Edit this dictionary to match your experiment.
# Keys are mouse names (matching pickle filenames without .pickle)
# ============================================================
control_h3_dict = {
    # example — replace with your actual mouse IDs and group labels:
    # '648L': 'ctr',
    # '649L': 'h3',
}

# If you haven't filled in the dict yet, print available names as a reminder
if not control_h3_dict:
    print('Fill in control_h3_dict above with your mouse IDs and group labels.')
    print('Available mouse names:', mouse_names)
else:
    print('Group assignments:', control_h3_dict)

In [ ]:
# Organise silhouette scores by group (run after filling in control_h3_dict)
ctr_h3_silh_dict = {'ctr': [], 'h3': []}
for mouse_name, group in control_h3_dict.items():
    score = silhouette_score_dict.get(mouse_name, np.nan)
    ctr_h3_silh_dict[group].append(score)
    print(f'{mouse_name}  ({group})  ->  {score:.4f}')

print('\nctr scores:', ctr_h3_silh_dict['ctr'])
print('h3  scores:', ctr_h3_silh_dict['h3'])

# Save the results (splitting control and h3)

In [ ]:
# Save results
file_to_save = pickle_dir + 'ctr_h3_silh_results.pickle'
results = {
    'silhouette_score_dict': silhouette_score_dict,
    'control_h3_dict': control_h3_dict,
    'ctr_h3_silh_dict': ctr_h3_silh_dict,
}
with open(file_to_save, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved -> {file_to_save}')

In [12]:
file_to_save = pickle_dir + 'ctr_h3_silh_results.pikcle'
results = {}
for i in ('ctr_h3_silh_dict',  
          'silhouette_score_dict',  
          'control_h3_dict'):
    results[i] = locals()[i]

with open(file_to_save, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [13]:
# sel_gm_list = list(itertools.product(range(1,3),range(1,n_mice+1)))
# sel_gm_list